# 06 · 최종 SVR 고정 재학습 및 결론

AutoML에서 정한 모델/피처/파라미터를 변경하지 않고 개발 37셀로 처음부터 다시 학습했다. Hold-out 9셀은 학습에 넣지 않았고 Batch 2·3도 보정/선정에 사용하지 않았다. 같은 데이터를 다시 돌린 재현성 점검이지 새로운 독립 검증은 아니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
A=ROOT/"results/final_svr"
summary=json.loads((A/"run_summary.json").read_text())
display(summary["frozen_parameters"])
display(summary["features"])
display(summary["checks"])
display(summary["max_abs_prediction_difference_vs_previous"])

**코멘트:** 입력은 DQ_relative_logvar 하나이고 중앙값 대치와 표준화를 학습 fold 내부에서만 fit한다. 타깃은 log(cycle_life), 예측은 exp로 복원한다. epsilon은 로그 수명 단위다. 독립 재학습 2회 일치, 저장 모델 재로드 일치, 타깃 제외, 그룹 fold 정책 분리 여부를 검증했다.

In [ ]:
display(pd.read_csv(A/"fixed_cv_summary.csv"))
display({"prior_nested_selection_MAPE":summary["previous_selection_procedure_nested_MAPE"], "prior_nested_selection_SD":summary["previous_selection_procedure_nested_SD"]})

**코멘트:** 위 4/5-fold는 이미 이 개발 데이터에서 선정한 파라미터를 고정해 계산한 CV로 낙관적일 수 있다. 새로운 독립 nested CV 성능이라고 보고하면 안 된다. 모델·피처·튜닝 전체 선택 절차를 평가한 기존 외부 CV 12.21% ±1.08pp를 함께 유지한다. 표준편차는 95% 신뢰구간이 아니다.

In [ ]:
display(pd.read_csv(A/"model_performance.csv"))
display(pd.read_csv(A/"performance_reporting.csv"))
display(Image(filename=str(A/"actual_vs_predicted.png")))

**코멘트:** train 재대입 MAPE를 CV 점수로 쓰지 않는다. 성능표의 Train(CV)는 이전 전체 선택 절차 nested CV이며 이번 재학습의 새로운 추정이 아니다. Batch 2 목표 9.1%는 원논문 데이터/분할/EOL이 같은지 미확인이라 참고 차이만 표시한다. 수명 없는 10셀은 예측만 하며 평가에서 제외한다.

In [ ]:
display(json.loads((A/"diagnosis.json").read_text()))
display(pd.read_csv(A/"error_analysis.csv").head(10))

## 최종 결론

선정 SVR의 재학습 성능은 이전 AutoML 결과를 재현했다. 현재 프로젝트 기준 모델로 유지하지만 Batch 2 약 30.32%로 목표 9.1%에 미달한다. Batch 2 대부분을 과대 예측해, 개발 데이터에 500 미만 수명 셀이 없는 점과 배치별 정책/실험 조건/label 차이가 원인 후보다. 이 진단은 인과 증명이 아니다. 초기 신호와 cycle_life 관계가 배치를 넘어 안정적인지는 아직 검증되지 않았다. 실제 ESS 운용/교체 시점 자동 결정에 배포할 수준으로 결론내리지 않는다. label/EOL·셀 포함/연결 기준을 우선 확인하고 별도 개발/평가 데이터를 확보하는 것이 다음 단계다. PDF 생성 없음.

In [ ]:
import joblib
from src.preprocess import load_data
df=load_data(ROOT/"data/battery_modeling_ready.parquet")
model=joblib.load(A/"final_model.joblib")
pred=model.predict(df)
assert np.isfinite(pred).all()
display(pd.DataFrame({"battery_id":df.battery_id,"predicted_cycle_life":pred}).head())

## Colab 재실행

프로젝트 ZIP을 풀고 루트에서 실행한다. 새로운 출력 폴더를 사용해 기존 결과를 보존한다. 기본 requirements로 고정 모델 학습이 가능하지만 이전 FLAML 모델과의 재현 비교를 위해 requirements-automl도 설치한다. joblib은 신뢰할 수 있는 프로젝트 파일만 로드한다.

In [ ]:
# %cd /content/ess-battery-project
# %pip install -r requirements-automl.txt
# !python -m src.final_svr --out results/final_svr_colab